# DriveMind AI — Model & Agent Evaluation

Loads the evaluation reports produced by `scripts/train_baseline.py`, `scripts/train_transformer.py`, and `scripts/evaluate.py` under `reports/evaluation/`.
If a report file is missing, the corresponding section explains how to generate it — no numbers are invented here.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

REPORTS = Path.cwd().parent / 'reports' / 'evaluation'

def load_report(name):
    path = REPORTS / name
    if not path.exists():
        print(f'{name} not found. Run the corresponding script first.')
        return None
    with open(path) as f:
        return json.load(f)

## Baseline vs Transformer comparison

In [ ]:
comparison = load_report('model_comparison.json')
if comparison:
    import pandas as pd
    rows = {name: {k: v for k, v in metrics.items() if k in
                    ('accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted')}
            for name, metrics in comparison.items()}
    pd.DataFrame(rows).T

## Agent-level evaluation

In [ ]:
agent_report = load_report('agent_evaluation.json')
if agent_report:
    for key in ['n_cases', 'intent_accuracy', 'tool_selection_accuracy',
                'entity_extraction_accuracy', 'task_success_rate',
                'safety_rejection_accuracy', 'fallback_rate', 'error_rate',
                'avg_latency_ms', 'p95_latency_ms']:
        print(f'{key}: {agent_report[key]}')
    if agent_report['failures']:
        print('\nFailures:')
        for f in agent_report['failures']:
            print(' -', f)

## Error analysis: real misclassified examples

In [ ]:
errors = load_report('error_analysis.json')
if errors:
    print(errors['note'])
    for e in errors['baseline_errors']:
        print(f"  text={e['text']!r} true={e['true_intent']} pred={e['predicted_intent']} difficulty={e['difficulty']}")

## Discussion

Observed error patterns (see printed examples above) are typically:
- Ambiguous phrasing that legitimately overlaps two intents (e.g. abbreviations colliding with an unrelated intent's vocabulary).
- Typo-perturbed examples (the dataset generator injects light spelling noise) occasionally flipping a directional word, which can invert intents like open/close.

These are genuine limitations of a template-generated dataset + linear TF-IDF classifier, not evaluation artifacts — see the README's 'Error Analysis' and 'Future Improvements' sections for how a transformer or larger dataset would be expected to address them.